In [ ]:
import langextract as lx
from rich.pretty import pprint

In [ ]:
# 1. Define the prompt and extraction rules
prompt = """
    Sos un asistente especializado en el análisis de documentos judiciales.
    Tu tarea es identificar y extraer menciones de información sensible para su posterior anonimización.
    Debés detectar fragmentos textuales que correspondan a cualquiera de las siguientes entidades:

    - "BANCO": Nombre de una entidad bancaria, pública o privada.
    - "CBU": Código Bancario Uniforme (22 dígitos) de una cuenta.
    - "CORREO_ELECTRONICO": Dirección de correo electrónico.
    - "CUIJ": Código Único de Identificación Jurídica de causas judiciales.
    - "CUIT_CUIL": Número de CUIT o CUIL de una persona física o jurídica.
    - "DIRECCION": Dirección postal específica (calle, número, etc.).
    - "DNI": Número de Documento Nacional de Identidad u otro documento identificatorio.
    - "EDAD": Edad explícita de una persona.
    - "ESTUDIOS": Nivel o institución educativa que permita identificar a la persona (ej. "primario incompleto", "secundario completo", "Licenciado en…").
    - "FECHA": Fecha completa o parcial (día, mes y/o año).
    - "LINK": Enlace o URL a una página web.
    - "LOC": Localización geográfica específica (ciudad, barrio, comisaría, etc.).
    - "MARCA_AUTOMOVIL": Marca de un vehículo (ej. Toyota, Ford).
    - "NACIONALIDAD": Nacionalidad de una persona (ej. "argentino", "brasileña").
    - "NUM_ACTUACION": Número identificatorio de una actuación administrativa o contravencional.
    - "NUM_CAJA_AHORRO": Número completo de una caja de ahorro o cuenta bancaria.
    - "NUM_EXPEDIENTE": Número de expediente judicial o administrativo.
    - "NUM_MATRICULA": Número de matrícula profesional o académica.
    - "PATENTE_DOMINIO": Patente o dominio de un vehículo.
    - "PER": Nombre y apellido(s) de una persona física. Los nombres inicializados y los apodos también cuentan como información sensible a anonimizar.
    - "TELEFONO": Número telefónico (fijo o celular).
"""

In [ ]:
# 2. Provide a high-quality example to guide the model
examples = [
    lx.data.ExampleData(
        text="El 5 de mayo de 2023 el señor Fiscal indicó que realizó distintas medidas de prueba y que del resultado surge que tanto la investigada como el menor Juan Pérez se domicilian en la calle Sarmiento 1234, de la localidad de Moreno, por lo que solicitó que se declare la incompetencia en razón del territorio y se envíe el caso al Juzgado de Garantías que corresponda del Departamento Judicial de Moreno, con jurisdicción en el partido de Moreno.",
        extractions=[
            lx.data.Extraction(
                extraction_class="FECHA", extraction_text="5 de mayo de 2023"
            ),
            lx.data.Extraction(extraction_class="PER", extraction_text="Juan Pérez"),
            lx.data.Extraction(
                extraction_class="DIRECCION", extraction_text="Sarmiento 1234"
            ),
            lx.data.Extraction(extraction_class="LOC", extraction_text="Moreno"),
        ],
    ),
    lx.data.ExampleData(
        text="JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVENCIONAL Y DE FALTAS N°10 SECRETARIA N°19\nCarlos Gómez sobre 84 - HOMICIDIO CULPOSO Y OTROS\nNúmero: 52345/2022\nCUIJ: 12-34567890-1\nActuación Nro: 2022-009876",
        extractions=[
            lx.data.Extraction(extraction_class="PER", extraction_text="Carlos Gómez"),
            lx.data.Extraction(
                extraction_class="NUM_EXPEDIENTE", extraction_text="52345/2022"
            ),
            lx.data.Extraction(
                extraction_class="CUIJ", extraction_text="12-34567890-1"
            ),
            lx.data.Extraction(
                extraction_class="NUM_ACTUACION", extraction_text="2022-009876"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Acusado: Miguel Torres, DNI 30123456, nacido el 14/02/1990, de 34 años de edad, de nacionalidad paraguaya, género varón cis, con estudios secundarios completos, hizo hasta 3er año porque fue padre joven, con último domicilio en Av. Corrientes 3456, de esta ciudad, donde vive con su hermana y su cuñado Jorge Pérez. Tiene dos hijos a su cargo, de 5 y 8 años. Su hijo de 8 vive con él, su hija de 5 vive con su madre, Laura Fernández.",
        extractions=[
            lx.data.Extraction(extraction_class="PER", extraction_text="Miguel Torres"),
            lx.data.Extraction(extraction_class="DNI", extraction_text="30123456"),
            lx.data.Extraction(extraction_class="FECHA", extraction_text="14/02/1990"),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="34"),
            lx.data.Extraction(
                extraction_class="NACIONALIDAD", extraction_text="paraguaya"
            ),
            lx.data.Extraction(
                extraction_class="ESTUDIOS",
                extraction_text="estudios secundarios completos",
            ),
            lx.data.Extraction(
                extraction_class="DIRECCION",
                extraction_text="Av. Corrientes 3456",
            ),
            lx.data.Extraction(extraction_class="PER", extraction_text="Jorge Pérez"),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="5"),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="8"),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Laura Fernández"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="El testigo Juan López dejó asentado su número de contacto: 11-2345-6789. Indicó que la médica Dra. Ana García, MN 12345, asistió al lugar donde se hallaba un vehículo Volkswagen, patente AB123CD.",
        extractions=[
            lx.data.Extraction(extraction_class="PER", extraction_text="Juan López"),
            lx.data.Extraction(
                extraction_class="TELEFONO", extraction_text="11-2345-6789"
            ),
            lx.data.Extraction(extraction_class="PER", extraction_text="Ana García"),
            lx.data.Extraction(
                extraction_class="NUM_MATRICULA", extraction_text="12345"
            ),
            lx.data.Extraction(
                extraction_class="MARCA_AUTOMOVIL",
                extraction_text="Volkswagen",
            ),
            lx.data.Extraction(
                extraction_class="PATENTE_DOMINIO", extraction_text="AB123CD"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Se identificó una transferencia bancaria con los siguientes datos: CUIT 20-12345678-3, CBU 2850590940090412345671, Caja de Ahorro N° 12345678, Banco Nación.",
        extractions=[
            lx.data.Extraction(
                extraction_class="CUIT_CUIL", extraction_text="20-12345678-3"
            ),
            lx.data.Extraction(
                extraction_class="CBU",
                extraction_text="2850590940090412345671",
            ),
            lx.data.Extraction(
                extraction_class="NUM_CAJA_AHORRO", extraction_text="12345678"
            ),
            lx.data.Extraction(
                extraction_class="BANCO", extraction_text="Banco Nación"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Para mayor información, comunicarse a fiscalia.central@justicia.gob.ar o visitar el sitio https://justicia.gob.ar/actuaciones.",
        extractions=[
            lx.data.Extraction(
                extraction_class="CORREO_ELECTRONICO",
                extraction_text="fiscalia.central@justicia.gob.ar",
            ),
            lx.data.Extraction(
                extraction_class="LINK",
                extraction_text="https://justicia.gob.ar/actuaciones",
            ),
        ],
    ),
    lx.data.ExampleData(
        text="En la Ciudad Autónoma de Buenos Aires, el día 5 de mayo de 2023, el Sr. Fiscal hace saber que Juan Pérez se domicilia en la calle Sarmiento 1234, localidad de Moreno.",
        extractions=[
            lx.data.Extraction(
                extraction_class="FECHA", extraction_text="5 de mayo de 2023"
            ),
            lx.data.Extraction(extraction_class="PER", extraction_text="Juan Pérez"),
            lx.data.Extraction(
                extraction_class="DIRECCION", extraction_text="Sarmiento 1234"
            ),
            lx.data.Extraction(extraction_class="LOC", extraction_text="Moreno"),
        ],
    ),
    lx.data.ExampleData(
        text="Por otra parte, la División Investigaciones Judiciales de la Policía Federal Argentina informó que no se dio intervención a ninguna otra Fiscalía u otro Juzgado por la sustracción del vehículo Volkswagen Voyage, dominio KXY-876",
        extractions=[
            lx.data.Extraction(
                extraction_class="PATENTE_DOMINIO", extraction_text="KXY-876"
            ),
            lx.data.Extraction(
                extraction_class="MARCA_AUTOMOVIL",
                extraction_text="Volkswagen Voyage",
            ),
        ],
    ),
    lx.data.ExampleData(
        text="A su vez, requirió informes al Banco BBVA Francés, respecto de las cuentas bancarias de la denunciante, Carla Alejandra Garcia, D.N.I. 36.998.621 identificadas como Caja de ahorro en pesos argentinos número 117-59824/6 con CBU 0180132640000004685591 y Caja de ahorro en dólares número 119-619018/2 con CBU 0170115544000062081822.",
        extractions=[
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Carla Alejandra Garcia"
            ),
            lx.data.Extraction(extraction_class="DNI", extraction_text="36.998.621"),
            lx.data.Extraction(
                extraction_class="NUM_CAJA_AHORRO", extraction_text="117-59824/6"
            ),
            lx.data.Extraction(
                extraction_class="CBU", extraction_text="0180132640000004685591"
            ),
            lx.data.Extraction(
                extraction_class="NUM_CAJA_AHORRO", extraction_text="119-619018/2"
            ),
            lx.data.Extraction(
                extraction_class="CBU", extraction_text="0170115544000062081822"
            ),
        ],
    ),
]

In [ ]:
len(examples)

In [ ]:
iter_examples = iter(examples)

In [ ]:
example = next(iter_examples)
pprint(example)

In [ ]:
# Sample text to extract from
text = "La Fiscalía determinó que el objeto de este caso es investigar el hecho que tuvo lugar el día 12 de marzo de 2023 a las 8:50 horas aproximadamente, ocasión en que Carlos Gómez y María Rodriguez estafaron a Juan Pérez por un monto total de pesos treinta y tres mil novecientos sesenta ($33.960)."

# Run the extraction
result = lx.extract(
    text_or_documents=text,
    prompt_description=prompt,
    examples=examples,
    model_id="llama3.1:8b",
    model_url="http://localhost:11434",
    max_char_buffer=16_384,
    fence_output=False,
    use_schema_constraints=False,
)

pprint(result)

In [ ]:
# Custom OpenAI model integration
# Support for Ollama gpt-oss:20b #116
# https://github.com/google/langextract/issues/116#issuecomment-3209618602
from langextract.providers.openai import OpenAILanguageModel


# Custom OpenAI model class to handle Ollama gpt-oss:20b
class CustomOpenAIModel(OpenAILanguageModel):
    def _process_single_prompt(self, prompt, config):
        api_params = {
            "model": self.model_id,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1,
            "max_tokens": 2000,
        }

        try:
            response = self._client.chat.completions.create(**api_params)
            output_text = response.choices[0].message.content
            return lx.inference.ScoredOutput(score=1.0, output=output_text)

        except Exception as e:
            raise lx.exceptions.InferenceRuntimeError(
                f"Custom OpenAI API error: {str(e)}", original=e
            ) from e


# Instantiate the custom model
custom_model = CustomOpenAIModel(
    model_id="gpt-oss:20b",
    api_key="dummy",
    base_url="http://localhost:11434/v1",
)

# Run extraction with the custom model
result = lx.extract(
    text_or_documents=text,
    prompt_description=prompt,
    examples=examples,
    model=custom_model,
    fence_output=False,
    use_schema_constraints=False,
)
pprint(result)